# Hoof Morphology Measurements in Racing Thoroughbreds Across Shoeing and Unshod Conditions Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a FAIR dataset using the `mlcroissant` library, specifically using the FAIR² dataset: **Hoof Morphology Measurements in Racing Thoroughbreds Across Shoeing and Unshod Conditions**.

### Dataset Source
The dataset source is provided via a Croissant schema (JSON-LD) URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.hjew-zxn9/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()  # Get dataset metadata as a dict
print(f"{metadata['name']}: {metadata['description']}")
print(f"\nLicense: {metadata['license']}")
print(f"Published: {metadata['datePublished']}")
print(f"Version: {metadata['version']}")

## 2. Data Overview
Review available record sets (`@id`), their fields (`@id`) and columns.

Each Croissant dataset can have one or more *record sets*. Each record set is a logical group of related records (like a table in a database or DataFrame). We'll list their IDs and fields.

In [ ]:
print("## Available record sets and their fields:\n")
for rs in dataset.record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    if 'name' in rs:
        print(f"  Name: {rs['name']}")
    print("  Fields:")
    for field in rs['fields']:
        print(f"    - Field @id: {field['@id']}, name: {field.get('name', '<unnamed>')}, datatype: {field.get('dataType', '<unspecified>')}")
    print("  Columns:")
    for col in rs.get('columns', []):
        print(f"    - Column @id: {col['@id']}, name: {col.get('name', '<unnamed>')}, datatype: {col.get('dataType', '<unspecified>')}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. We use the record set and field `@id`s from the overview above.

Let's extract all record sets defined in this dataset.

In [ ]:
record_sets = [rs['@id'] for rs in dataset.record_sets]
if not record_sets:
    raise ValueError("No record sets found in metadata. Check the schema or Croissant definition.")

# Load all record sets as DataFrames
dataframes = {}
for record_set_id in record_sets:
    print(f"\nLoading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id, download=True))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"- Columns: {df.columns.tolist()}")
    print(f"- # Rows: {len(df)}")
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Apply basic data analysis such as filtering (e.g. for a hoof measurement > threshold), normalization, and grouping for a numeric column in a record set.

For demonstration, we select the first record set and a numeric field (e.g., 'ToeAngle_pre') if present.

In [ ]:
# Use the first available record set
primary_record_set_id = record_sets[0]
df = dataframes[primary_record_set_id]

# Attempt to automatically select a numeric field (prefer common morphometric fields)
numeric_field_candidates = [col for col in df.columns if any(key in col.lower() for key in ['angle', 'width', 'length', 'height'])]
if len(numeric_field_candidates) > 0:
    numeric_field = numeric_field_candidates[0]
else:
    # fallback: just pick first numeric field
    numeric_field = df.select_dtypes(include='number').columns[0] if len(df.select_dtypes(include='number').columns) > 0 else df.columns[0]

# Filter: values > mean (can adjust)
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10

filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (N={len(filtered_df)})")
display_cols = [numeric_field]
print(filtered_df[display_cols].head())

# Normalize
if pd.api.types.is_numeric_dtype(df[numeric_field]):
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping (e.g. by horse_id or condition)
group_field_candidates = [col for col in df.columns if any(k in col.lower() for k in ['horse', 'id', 'condition', 'group'])]
group_field = group_field_candidates[0] if group_field_candidates else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nMeans of {numeric_field} grouped by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field. We'll plot a histogram and, if grouping is available, plot mean values by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
plt.title(f"Distribution of {numeric_field} in RecordSet {primary_record_set_id}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot by group if applicable
if group_field:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
We used the `mlcroissant` library to load, display, filter, and visualize a tabular FAIR² dataset describing hoof morphometrics in Thoroughbred racing horses under different shoeing conditions. 

- You can adapt the fields and groupings above to focus on any available numeric or categorical column.
- For more details on dataset variables, refer to the Croissant schema documentation and field/column `@id`s for robust programmatic access.

Further steps might include hypothesis testing, modeling, or longitudinal visualization by dates or repeated measures.